<!-- course-header -->
# Lesson 15: Agents, tool use and MCP

*Worked solutions for lesson 15. Read the lesson [online](https://ashutoshjoshi1.github.io/AI-ZerotoMastery/lessons/15-agents.html) or [offline](../../lessons/15-agents.html). The exercise version without solutions is `../15-agents.ipynb`.*

<a href="https://colab.research.google.com/github/ashutoshjoshi1/AI-ZerotoMastery/blob/main/notebooks/solutions/15-agents-solutions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Give models the ability to act by writing the tool-use loop by hand, designing tools models use well, choosing between workflows and autonomous agents, and building an MCP server, then add the guardrails and evaluation that make agents safe to deploy.

**Runs on:** Any computer, plus an API key for the agent loop.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Each exercise has a cell titled *Hands-on* with the same name as in the lesson. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q anthropic python-dotenv "mcp[cli]"

In [ ]:
from dotenv import load_dotenv
load_dotenv()

## Tool use: the mechanism

### Define tools

In [ ]:
import json, math, ast, operator as op
import anthropic

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"

# --- the actual tool implementations -------------------------------------------
FLEET = {   # stand-in for a real database or API
    "P-117": {"site": "coastal", "status": "offline", "last_data": "2026-10-18T02:00Z", "firmware": "2.3"},
    "P-204": {"site": "urban", "status": "ok", "last_data": "2026-10-19T09:55Z", "firmware": "2.4"},
}

def get_instrument_status(instrument_id: str) -> dict:
    if instrument_id not in FLEET:
        return {"error": f"unknown instrument '{instrument_id}'. Known ids: {sorted(FLEET)}"}
    return FLEET[instrument_id]

OPS = {ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
       ast.Pow: op.pow, ast.Mod: op.mod, ast.USub: op.neg, ast.UAdd: op.pos}
FUNCS = {"sqrt": math.sqrt, "log": math.log, "exp": math.exp, "abs": abs, "round": round}

def calculate(expression: str) -> float:
    """Evaluate arithmetic safely: never call eval() on model output."""
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS:
            left, right = ev(node.left), ev(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 100:
                raise ValueError("exponent too large")
            return OPS[type(node.op)](left, right)
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS:
            return OPS[type(node.op)](ev(node.operand))
        if isinstance(node, ast.Call) and isinstance(node.func, ast.Name) and node.func.id in FUNCS:
            return FUNCS[node.func.id](*[ev(a) for a in node.args])
        if isinstance(node, ast.Name) and node.id in ("pi", "e"):
            return getattr(math, node.id)
        raise ValueError("unsupported expression")
    return ev(ast.parse(expression, mode="eval").body)

# --- what the model sees ---------------------------------------------------------
TOOLS = [
    {
        "name": "get_instrument_status",
        "description": ("Look up the current status of one field instrument by its id (format 'P-123'). "
                        "Returns site, status (ok/degraded/offline), time of last data (UTC) and firmware version."),
        "input_schema": {
            "type": "object",
            "properties": {"instrument_id": {"type": "string", "description": "Instrument id, e.g. 'P-117'"}},
            "required": ["instrument_id"],
        },
    },
    {
        "name": "calculate",
        "description": "Evaluate an arithmetic expression exactly. Use for any calculation instead of mental arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "e.g. '(17 - 2.5) * 3600'"}},
            "required": ["expression"],
        },
    },
]
TOOL_FUNCS = {"get_instrument_status": get_instrument_status, "calculate": calculate}

In [ ]:
print(get_instrument_status("P-117"))
print(get_instrument_status("P117"))          # the error message tells the model how to fix its call
print(calculate("(24 + 8) * 60"))
try:
    calculate("__import__('os').system('ls')")
except ValueError as e:
    print("blocked:", e)

### The loop

In [ ]:
# Needs an API key (see the setup cell)
def run_agent(task: str, max_steps: int = 10, verbose: bool = True) -> str:
    messages = [{"role": "user", "content": task}]
    for step in range(max_steps):
        resp = client.messages.create(
            model=MODEL, max_tokens=1500, tools=TOOLS, messages=messages,
            system="You are an operations assistant for a fleet of field instruments. "
                   "Use tools to check facts rather than guessing. Be concise.",
        )
        messages.append({"role": "assistant", "content": resp.content})   # keep the tool calls in history

        if resp.stop_reason != "tool_use":                                 # the model has finished
            return "".join(b.text for b in resp.content if b.type == "text")

        results = []
        for block in resp.content:                                         # may contain several calls
            if block.type != "tool_use":
                continue
            if verbose:
                print(f"  -> {block.name}({block.input})")
            try:
                output = TOOL_FUNCS[block.name](**block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": json.dumps(output)})
            except Exception as e:                                         # report errors to the model
                results.append({"type": "tool_result", "tool_use_id": block.id,
                                "content": f"Error: {e}", "is_error": True})
        messages.append({"role": "user", "content": results})              # results go back as a user turn
    return "Stopped: step limit reached before the task was finished."

print(run_agent("Is P-117 healthy? If not, how many hours has it been without data as of 2026-10-19T10:00Z?"))
#   -> get_instrument_status({'instrument_id': 'P-117'})
#   -> calculate({'expression': '24 + 8'})
# P-117 is offline. Its last data arrived at 02:00 UTC on 18 October, so it has been
# without data for 32 hours as of 10:00 UTC today.

## The Model Context Protocol

### Build a server

Run in a terminal:

```bash
uv add "mcp[cli]"
```

In [ ]:
%%writefile fleet_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("instrument-fleet")

FLEET = {
    "P-117": {"site": "coastal", "status": "offline", "last_data": "2026-10-18T02:00Z", "firmware": "2.3"},
    "P-204": {"site": "urban", "status": "ok", "last_data": "2026-10-19T09:55Z", "firmware": "2.4"},
}

@mcp.tool()
def get_instrument_status(instrument_id: str) -> dict:
    """Current status of one instrument by id (format 'P-123'): site, status, last data time (UTC), firmware."""
    return FLEET.get(instrument_id, {"error": f"unknown id; known ids: {sorted(FLEET)}"})

@mcp.tool()
def list_instruments(status: str | None = None) -> list[str]:
    """List instrument ids, optionally only those with a given status ('ok', 'degraded', 'offline')."""
    return [i for i, v in FLEET.items() if status is None or v["status"] == status]

@mcp.resource("fleet://firmware-notes")
def firmware_notes() -> str:
    """Release notes for instrument firmware versions."""
    return "2.4: fixes overnight logging stall on cellular modems. 2.3: adds GPS time sync."

if __name__ == "__main__":
    mcp.run()          # speaks MCP over stdin/stdout by default

The decorators turn type hints and docstrings into the tool schemas and descriptions, so write them carefully: they are what the model reads. Test the server interactively with the MCP Inspector (`uv run mcp dev fleet_server.py`), then register it with a client. For Claude Desktop, add it to the configuration file:

*claude_desktop_config.json*

```json
{
  "mcpServers": {
    "fleet": {
      "command": "uv",
      "args": ["run", "--directory", "/path/to/project", "fleet_server.py"]
    }
  }
}
```

> **MCP servers are code you run with your permissions**
>
> Only install servers you trust, give them the narrowest credentials that work, and remember that tool results are untrusted input to the model: a document or web page fetched by one tool can contain instructions trying to hijack the agent (prompt injection, lesson 22).

## Guardrails, budgets and evaluation

### Hands-on: a research agent over your documents

Combine lessons 14 and 15.

1. Wrap your `MiniRAG.retrieve` as a tool `search_docs(query: str, k: int = 5)` returning chunk ids, titles and text.
2. Add a tool `read_chunk(chunk_id: str)` that returns a chunk with its neighbours for more context, and the `calculate` tool.
3. Give the agent a system prompt that requires citations by chunk id and allows "not found".
4. Add limits: 12 steps, and a running token count that stops the agent above 60,000 input tokens.
5. Write 8 questions that need *several* searches (comparisons, multi-step lookups) and compare the agent's answers, cost and latency with plain `MiniRAG.answer`.

#### Solution: a solution sketch and typical findings

In [ ]:
# Uses your own files: edit the paths or data first; Downloads a model or data on first run; Needs an API key (see the setup cell)
# First copy chunk_text, BM25, rrf, RAG_PROMPT, format_sources and MiniRAG from the lesson 14 notebook,
# and set `documents` to your collection.
rag = MiniRAG(); rag.index(documents)
by_id = {c["id"]: i for i, c in enumerate(rag.chunks)}

def search_docs(query: str, k: int = 5) -> list[dict]:
    return [{"id": c["id"], "title": c["title"], "text": c["text"][:1200]}
            for c in rag.retrieve(query, k_final=min(k, 8))]

def read_chunk(chunk_id: str) -> dict:
    i = by_id.get(chunk_id)
    if i is None:
        return {"error": "unknown chunk id; use ids returned by search_docs"}
    window = rag.chunks[max(0, i - 1): i + 2]
    return {"id": chunk_id, "text": "\n\n".join(c["text"] for c in window)}

TOOL_FUNCS.update(search_docs=search_docs, read_chunk=read_chunk)
TOOLS += [
    {"name": "search_docs",
     "description": "Search the document collection. Returns the most relevant chunks with ids. "
                    "Search several times with different wording for multi-part questions.",
     "input_schema": {"type": "object",
                      "properties": {"query": {"type": "string"}, "k": {"type": "integer", "default": 5}},
                      "required": ["query"]}},
    {"name": "read_chunk",
     "description": "Read a chunk together with its neighbouring chunks, for more context.",
     "input_schema": {"type": "object", "properties": {"chunk_id": {"type": "string"}},
                      "required": ["chunk_id"]}},
]
# In run_agent: accumulate resp.usage.input_tokens and stop when the total exceeds 60_000.

Typical findings: on simple lookups the agent matches plain RAG but costs several times more and takes longer. On comparison and multi-hop questions it often wins clearly, because it searches for each part separately and follows up on what it finds. Failures to look for: repeating near-identical searches, citing chunks it never read, and stopping too early. Each suggests a fix to the tool descriptions or system prompt. The cost and latency gap is why production systems often route: plain RAG first, the agent only for questions that need it.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/15-agents.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.